[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuelcostac/generative-networks/blob/main/pretraining/cpc.ipynb)

This notebook is part of the [Generative Networks course](https://www.nahuelcosta.com/teaching/generative-networks).

# Contrastive Predictive Coding on Image Rows

We adapt CPC to images by encoding an 8×8 patch grid, reading its rows from top to bottom, and asking an autoregressive context model to identify future latent rows among in-batch alternatives.

The model in this notebook is deliberately small. It demonstrates a pretraining objective and representation-transfer workflow; it is **not** a foundation model. Foundation models additionally require broad data, much larger capacity, and extensive compute.


CPC is simultaneously predictive, autoregressive, contrastive, and latent-space based. This teaching adaptation makes the temporal axis explicit by treating image rows as a sequence. It does not reconstruct pixels: each prediction head must select the correct future row representation for horizons one, two, and three.


## 0. Dependencies


In [ ]:
from importlib import util
import subprocess
import sys

REQUIRED = {"torch": "torch", "torchvision": "torchvision", "matplotlib": "matplotlib", "numpy": "numpy"}
missing = [package for module, package in REQUIRED.items() if util.find_spec(module) is None]
if missing:
    print("Installing missing course dependencies:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Course dependencies are already installed.")


## 1. Imports and Reproducibility


In [ ]:
import copy
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fast_dev_run = os.environ.get("FAST_DEV_RUN", "0") == "1"
print(f"Using device: {device}; FAST_DEV_RUN={fast_dev_run}")


## 2. Configuration


In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "pretraining" else Path.cwd()
output_dir = project_root / "pretraining" / "outputs" / "cpc"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / "cpc_encoder.pt"
patch_size, grid_size = 4, 8
latent_dim, context_dim = 128, 256
horizons = (1, 2, 3)
temperature = 0.1
batch_size = 64 if fast_dev_run else 256
epochs = 1 if fast_dev_run else 20
max_steps = 2 if fast_dev_run else None
learning_rate = 1e-3
print("Artifacts:", output_dir)


## 3. CIFAR-10 as a Sequence of Patch Rows


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
data_root = project_root / "data"

evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
raw_train = datasets.CIFAR10(data_root, train=True, download=True)
probe_source = datasets.CIFAR10(data_root, train=True, transform=evaluation_transform)
test_source = datasets.CIFAR10(data_root, train=False, download=True, transform=evaluation_transform)

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(raw_train), generator=generator).tolist()
pretrain_count = 256 if fast_dev_run else 20_000
probe_count = 256 if fast_dev_run else 10_000
test_count = 256 if fast_dev_run else len(test_source)
pretrain_indices = permutation[:pretrain_count]
probe_indices = permutation[:probe_count]

probe_train = Subset(probe_source, probe_indices)
probe_test = Subset(test_source, range(test_count))
loader_kwargs = {"num_workers": 0 if fast_dev_run else 2, "pin_memory": device.type == "cuda"}
probe_train_loader = DataLoader(probe_train, batch_size=256, shuffle=False, **loader_kwargs)
probe_test_loader = DataLoader(probe_test, batch_size=256, shuffle=False, **loader_kwargs)

print(f"Unlabeled pretraining images: {pretrain_count:,}")
print(f"Labeled probe images: {probe_count:,}; test images: {test_count:,}")


In [ ]:
pretrain_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])
pretrain_source = datasets.CIFAR10(data_root, train=True, transform=pretrain_transform)
pretrain_loader = DataLoader(
    Subset(pretrain_source, pretrain_indices), batch_size=batch_size, shuffle=True,
    drop_last=True, generator=torch.Generator().manual_seed(SEED), **loader_kwargs,
)

def patch_rows(images):
    batch, channels, height, width = images.shape
    patches = images.reshape(batch, channels, grid_size, patch_size, grid_size, patch_size)
    return patches.permute(0, 2, 4, 1, 3, 5).reshape(batch, grid_size, grid_size, -1)

images, _ = next(iter(pretrain_loader))
rows = patch_rows(images)
assert rows.shape == (images.size(0), 8, 8, 48)
print("Patch grid:", rows.shape, "-> eight ordered row observations")


## 4. Local Encoder, Autoregressive Context, and Predictors

A shared MLP encodes each local patch. A row projection preserves the ordered set of eight patches. The GRU state at row `t` can depend only on rows up to `t`; separate linear heads predict different future horizons.


In [ ]:
class ImageCPC(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_encoder = nn.Sequential(nn.Linear(48, latent_dim), nn.ReLU(), nn.Linear(latent_dim, latent_dim))
        self.row_projection = nn.Sequential(nn.Linear(grid_size * latent_dim, latent_dim), nn.ReLU())
        self.context = nn.GRU(latent_dim, context_dim, batch_first=True)
        self.predictors = nn.ModuleDict({str(k): nn.Linear(context_dim, latent_dim) for k in horizons})

    def encode_rows(self, images):
        patches = patch_rows(images)
        local = self.patch_encoder(patches)
        return self.row_projection(local.flatten(2))

    def contexts(self, images):
        rows = self.encode_rows(images)
        contexts, _ = self.context(rows)
        return rows, contexts

    def forward(self, images):
        rows, contexts = self.contexts(images)
        losses, accuracies = [], []
        batch = images.size(0)
        for horizon in horizons:
            predictions = F.normalize(self.predictors[str(horizon)](contexts[:, :-horizon]), dim=-1)
            targets = F.normalize(rows[:, horizon:], dim=-1)
            # For each time step, candidates are future rows from every image in the batch.
            logits = torch.einsum("btd,ctd->tbc", predictions, targets) / temperature
            labels = torch.arange(batch, device=images.device).repeat(logits.size(0))
            losses.append(F.cross_entropy(logits.reshape(-1, batch), labels))
            accuracies.append((logits.argmax(-1) == torch.arange(batch, device=images.device)).float().mean())
        return torch.stack(losses).mean(), torch.stack(accuracies), rows, contexts

    def forward_features(self, images):
        return self.contexts(images)[1][:, -1]

model = ImageCPC().to(device)
loss, accuracies, rows, contexts = model(images[:8].to(device))
assert rows.shape == (8, 8, latent_dim) and contexts.shape == (8, 8, context_dim)
loss.backward()
assert math.isfinite(loss.item()) and model.context.weight_ih_l0.grad is not None
model.zero_grad(set_to_none=True)
print("Initial loss:", loss.item(), "retrieval by horizon:", accuracies.detach().cpu().tolist())


## 5. Verify Causality Behaviorally


In [ ]:
model.eval()
original = images[:4].to(device)
changed = original.clone()
changed[:, :, 16:, :] = torch.randn_like(changed[:, :, 16:, :])
with torch.no_grad():
    _, context_original = model.contexts(original)
    _, context_changed = model.contexts(changed)
assert torch.equal(context_original[:, :4], context_changed[:, :4])
assert not torch.equal(context_original[:, 4:], context_changed[:, 4:])
print("Changing future pixel rows cannot alter earlier context states.")


## 6. Pretrain CPC


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
history, retrieval_history, step = [], [], 0
for epoch in range(epochs):
    model.train(); running_loss, running_accuracy = [], []
    for images, _ in pretrain_loader:
        images = images.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            loss, accuracies, *_ = model(images)
        optimizer.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        running_loss.append(loss.item()); running_accuracy.append(accuracies.detach().cpu().numpy()); step += 1
        if max_steps is not None and step >= max_steps: break
    history.append(float(np.mean(running_loss)))
    retrieval_history.append(np.mean(running_accuracy, axis=0).tolist())
    print(f"Epoch {epoch + 1:02d}/{epochs:02d} | InfoNCE={history[-1]:.4f} | retrieval={retrieval_history[-1]}")
    if max_steps is not None and step >= max_steps: break

torch.save({"model": model.state_dict(), "history": history, "retrieval": retrieval_history}, checkpoint_path)
assert np.isfinite(history).all() and np.isfinite(retrieval_history).all()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history, marker="o"); axes[0].set(title="CPC pretraining", xlabel="Epoch", ylabel="InfoNCE")
for index, horizon in enumerate(horizons):
    axes[1].plot([row[index] for row in retrieval_history], label=f"k={horizon}")
axes[1].set(title="Future-row retrieval", xlabel="Epoch", ylabel="Accuracy"); axes[1].legend()
for axis in axes: axis.grid(alpha=0.3)
plt.tight_layout()


## 7. Frozen Linear Probe


In [ ]:
@torch.no_grad()
def extract_features(encoder, loader):
    encoder.eval()
    features, labels = [], []
    for images, targets in loader:
        features.append(encoder(images.to(device)).float().cpu())
        labels.append(targets)
    return torch.cat(features), torch.cat(labels)

def fit_linear_probe(encoder, name):
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    train_x, train_y = extract_features(encoder, probe_train_loader)
    test_x, test_y = extract_features(encoder, probe_test_loader)
    head = nn.Linear(train_x.size(1), 10).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
    train_loader = DataLoader(
        TensorDataset(train_x, train_y), batch_size=256, shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    epochs = 1 if fast_dev_run else 15
    history = []
    for epoch in range(epochs):
        head.train()
        for features, targets in train_loader:
            logits = head(features.to(device))
            loss = F.cross_entropy(logits, targets.to(device))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.no_grad():
            predictions = head(test_x.to(device)).argmax(1).cpu()
        accuracy = (predictions == test_y).float().mean().item()
        history.append(accuracy)
    print(f"{name}: final frozen-probe accuracy={accuracy:.3f}")
    return {"name": name, "accuracy": accuracy, "history": history}


In [ ]:
class CPCFeatureEncoder(nn.Module):
    def __init__(self, cpc):
        super().__init__(); self.cpc = cpc
    def forward(self, images):
        return self.cpc.forward_features(images)

random_encoder = CPCFeatureEncoder(ImageCPC())
pretrained_encoder = CPCFeatureEncoder(model)


In [ ]:
probe_results = [
    fit_linear_probe(random_encoder.to(device), "Random frozen encoder"),
    fit_linear_probe(pretrained_encoder.to(device), "Pretrained frozen encoder"),
]
plt.figure(figsize=(6, 4))
plt.bar([row["name"] for row in probe_results], [row["accuracy"] for row in probe_results])
plt.ylabel("CIFAR-10 test accuracy")
plt.xticks(rotation=15, ha="right")
plt.ylim(0, 1)
plt.title("Representation quality through a linear probe")
plt.tight_layout()


## 8. Reload the Checkpoint


In [ ]:
reloaded = ImageCPC().to(device)
reloaded.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True)["model"])
reloaded.eval(); model.eval(); check = images[:4].to(device)
with torch.no_grad():
    assert torch.equal(model.forward_features(check), reloaded.forward_features(check))
print("Reloaded CPC representations match exactly.")


## 9. Things to Try

- Predict columns, diagonal traversals, or a rasterized sequence instead of rows.
- Change the maximum horizon and inspect retrieval difficulty.
- Draw negatives from other time positions as well as other images.
- Replace the GRU with a causal Transformer.

## Troubleshooting

- Retrieval near `1 / batch_size` is chance performance.
- A smaller batch reduces both memory use and the number of negative candidates.
- This image-row construction is a transparent teaching adaptation, not a reproduction of the paper's full vision architecture.

**Reference:** [Representation Learning with Contrastive Predictive Coding](https://arxiv.org/abs/1807.03748).
